# Notebook 02 - Grafo de Transicoes

Explore o grafo: sucessores, predecessores, posicao tipica e transicoes mais frequentes.

> Rode apos ter coletado pelo menos 10 sets no notebook 01.

In [ ]:
import sys
sys.path.insert(0, '../')

import os
from dotenv import load_dotenv
from supabase import create_client
from src.graph.graph import Graph
import pandas as pd

load_dotenv('../.env')
sb    = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_KEY'])
graph = Graph(sb)

generos    = sb.table('genres').select('id, slug, name').eq('active', True).execute().data
genre_map  = {g['slug']: g for g in generos}
print('Grafo pronto')
print('Generos disponiveis:', [g['slug'] for g in generos])

## Estatisticas por genero

In [ ]:
rows = []
for g in generos:
    st = graph.get_genre_stats(g['id'])
    rows.append({'Genero': g['name'], 'Sets': st['sets'], 'Transicoes': st['transitions']})
df = pd.DataFrame(rows).sort_values('Sets', ascending=False)
print(f'Total: {df["Sets"].sum()} sets | {df["Transicoes"].sum()} transicoes')
display(df)

## Top 15 tracks que seguem uma track

In [ ]:
# -- EDITE AQUI --
ARTISTA = 'Astrix'
TITULO  = 'Basic'
GENERO  = 'goa-psy-trance'
# ----------------

genre    = genre_map.get(GENERO)
track_id = graph.find_track_id(ARTISTA, TITULO)

if not track_id:
    print(f'Track nao encontrada na base: {ARTISTA} - {TITULO}')
    print('Rode o notebook 01 para coletar mais sets primeiro.')
else:
    succs = graph.get_successors(track_id, genre['id'], top_n=15)
    print(f'Top {len(succs)} sucessores de "{ARTISTA} - {TITULO}" em {genre["name"]}:')
    print('-' * 60)
    for i, s in enumerate(succs, 1):
        bpm = f"BPM {s.get('bpm')}" if s.get('bpm') else ''
        key = f"Key {s.get('camelot_key')}" if s.get('camelot_key') else ''
        print(f"{i:02d}. {s.get('artist')} - {s.get('title')} ({s.get('count')}x) {bpm} {key}")

## Posicao tipica de uma track nos sets

In [ ]:
# -- EDITE AQUI --
ARTISTA = 'Vini Vici'
TITULO  = 'The Tribe'
# ----------------

track_id = graph.find_track_id(ARTISTA, TITULO)
if track_id:
    pos = graph.get_typical_position(track_id)
    zonas = {
        'opening':  'Abertura (0-20%)',
        'buildup':  'Construcao (20-45%)',
        'peak':     'Pico (45-70%)',
        'comedown': 'Descida (70-85%)',
        'closing':  'Fechamento (85-100%)',
        'middle':   'Meio (base insuficiente)',
    }
    print(f'{ARTISTA} - {TITULO}')
    print(f'  Posicao media: {pos["avg_position_pct"]*100:.0f}% do set')
    print(f'  Zona: {zonas.get(pos["typical_zone"], pos["typical_zone"])}')
else:
    print(f'Track nao encontrada: {ARTISTA} - {TITULO}')

## Score de uma transicao especifica

In [ ]:
# -- EDITE AQUI --
FROM_ARTISTA = 'Astrix'
FROM_TITULO  = 'Basic'
TO_ARTISTA   = 'Vini Vici'
TO_TITULO    = 'The Tribe'
GENERO       = 'goa-psy-trance'
# ----------------

genre   = genre_map.get(GENERO)
id_from = graph.find_track_id(FROM_ARTISTA, FROM_TITULO)
id_to   = graph.find_track_id(TO_ARTISTA, TO_TITULO)

if id_from and id_to:
    count = graph.get_transition_count(id_from, id_to, genre['id'])
    print(f'"{FROM_ARTISTA} - {FROM_TITULO}" -> "{TO_ARTISTA} - {TO_TITULO}"')
    print(f'Ocorrencias em sets reais de {genre["name"]}: {count}')
    if   count == 0: print('Nunca vista em sets reais desse genero')
    elif count <  3: print('Rara, mas acontece')
    elif count < 10: print('Transicao valida')
    else:            print(f'Transicao popular! ({count}x)')
else:
    print('Uma ou ambas as tracks nao estao na base ainda.')

## Top 20 tracks mais ativas por genero

In [ ]:
from collections import Counter

# -- EDITE AQUI --
GENERO = 'goa-psy-trance'
# ----------------

genre = genre_map.get(GENERO)
trans = sb.table('transitions').select('track_from_id').eq('genre_id', genre['id']).execute().data

if not trans:
    print('Nenhuma transicao na base para esse genero. Rode o notebook 01 primeiro.')
else:
    counts = Counter(t['track_from_id'] for t in trans)
    print(f'Top 20 em {genre["name"]}:')
    print('-' * 55)
    for i, (tid, cnt) in enumerate(counts.most_common(20), 1):
        t = graph.get_track_data(tid)
        if t:
            print(f"{i:02d}. {t['artist']} - {t['title']} ({cnt}x)")